# Day 1: Instructor solution to the representation exercise

This notebook is separate from the student notebook. Run it top to bottom. It uses the same corpus and vectorizer settings, runs all three representation choices, and then compares time periods. The Markdown cells contain the observed results and worked discussion so they are visible before executing code.

The recorded numbers below were produced with the blank English spaCy tokenizer available in the current environment; results may differ slightly with another spaCy or scikit-learn version.

In [ ]:
# Import tools for tables, visualizations, text features, and vector comparisons.
import re
from pathlib import Path

# NumPy handles arrays and matrix totals; pandas holds labeled tables.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# scikit-learn supplies text-to-matrix tools, dimensionality reduction, and similarity.
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Show longer text snippets in tables without truncating them too early.
pd.set_option('display.max_colwidth', 120)

# Find the local data folder whether Jupyter starts in the repository or notebook directory.
from pathlib import Path


# Jupyter may start in the repository root or in materials/notebooks.
def find_data_dir():
    # Try the likely relative paths in order; Path joins folders across operating systems.
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        # Use the first folder that is present on this computer.
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')


# Keep the chosen folder in one variable for all later data loads.
DATA_DIR = find_data_dir()
DATA_DIR


In [ ]:
# Load a spaCy pipeline and define the token rules used throughout this notebook.
import spacy

# The trained model adds tags, lemmas, parsing, and named entities.
try:
    nlp = spacy.load('en_core_web_sm')
    print('Loaded spaCy model: en_core_web_sm')
except OSError:
    # Fallback: a blank English pipeline still tokenizes text, but has no trained annotations.
    nlp = spacy.blank('en')
    nlp.add_pipe('sentencizer')
    print('Using spacy.blank("en"). Install en_core_web_sm for POS tags, lemmas, dependency parses, and named entities.')


# These switches decide which tokens will be eligible as word features.
# Inputs: raw text and three switches for stop words, nonletters, and short tokens.
# Output: a list of lowercased word features used by the vectorizers below.
def spacy_preprocess(text, remove_stop=True, keep_alpha=True, min_len=2):
    """Tokenize and lightly preprocess text with spaCy."""
    # make_doc tokenizes only; it does not run the model's slower tagging/parsing steps.
    doc = nlp.make_doc(str(text))
    tokens = []
    for token in doc:
        # Skip numbers and punctuation when keep_alpha=True.
        if keep_alpha and not token.is_alpha:
            continue
        # Skip spaCy's common-word list when remove_stop=True.
        if remove_stop and token.is_stop:
            continue
        # Lowercasing makes 'Policy' and 'policy' the same feature.
        term = token.text.lower()
        if len(term) >= min_len:
            tokens.append(term)
    return tokens


# CountVectorizer calls this function once per document to obtain its features.
def spacy_analyzer(text):
    return spacy_preprocess(text)


# This alternate analyzer returns both the usual unigrams and two-word features.
def spacy_analyzer_with_bigrams(text):
    """Keep the usual unigrams and add adjacent word pairs."""
    unigrams = spacy_preprocess(text)
    # Keep the original token order so a bigram really was adjacent in the text.
    original_tokens = list(nlp.make_doc(str(text)))
    # Make pairs before stop-word removal: otherwise distant words can
    # become false neighbors and phrases such as 'not good' disappear.
    # zip(tokens, tokens[1:]) walks over neighboring pairs; an underscore joins each pair into one feature.
    bigrams = [
        left.text.lower() + '_' + right.text.lower()
        for left, right in zip(original_tokens, original_tokens[1:])
        if left.is_alpha and right.is_alpha
        and len(left.text) >= 2 and len(right.text) >= 2
    ]
    # The same document can contribute single words and word pairs to one matrix.
    return unigrams + bigrams


# Return one row per token to reveal what spaCy recognized and what preprocessing keeps.
def spacy_token_table(text):
    """Show tokenization, preprocessing flags, and parses when a full model is available."""
    # Unlike make_doc, calling nlp runs available model components for tags and entities.
    doc = nlp(str(text))
    rows = []
    for token in doc:
        rows.append({
            'text': token.text,
            'lemma': token.lemma_ or '(model needed)',
            'pos': token.pos_ or '(model needed)',
            'dep': token.dep_ or '(model needed)',
            'head': token.head.text if token.dep_ else '(model needed)',
            'ent_type': token.ent_type_ or '',
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'like_num': token.like_num,
            # Apply our default word rules to this token so the last column explains feature selection.
            'kept_by_preprocess': token.text.lower() in spacy_preprocess(token.text, remove_stop=True)
        })
    return pd.DataFrame(rows)


In [ ]:
# Load the State of the Union corpus and convert its year field to a number.
# Load the supplied State of the Union file; one row represents one speech.
sotu = pd.read_csv(DATA_DIR / 'sotu.csv')
# Invalid or missing years become NaN, which we remove before filtering by year.
sotu['year_numeric'] = pd.to_numeric(sotu['year'], errors='coerce')

sotu[['president', 'year', 'party', 'sotu_type', 'text']].head()

# Keep modern speeches from the two parties used in the group comparison.
modern = (
    sotu
    # Remove rows that lack the fields needed for the year, party, or text comparisons.
    .dropna(subset=['year_numeric', 'party', 'text'])
    # Hold the analysis to modern speeches from the two parties used below.
    .query("party in ['Democratic', 'Republican'] and year_numeric >= 1950")
    # Work on an independent DataFrame so adding columns does not modify sotu.
    .copy()
)

# Make short labels for plots and speech-pair tables.
modern['doc_label'] = modern['president'] + ' (' + modern['year'].astype(str) + ')'
modern[['doc_label', 'party', 'sotu_type']].head(), modern.shape


## Answer key for the new exercise scaffold

The [student exercise scaffold](day_1_representation_exercise_scaffold.ipynb) asks students to complete three code edits. The cells below run **all three choices**; each uses the same five completed lines for fitting, vocabulary comparison, and similarity.

| Choice | `student_analyzer` | `student_min_df` | `student_binary` |
| --- | --- | ---: | --- |
| A: rarer terms removed | `spacy_analyzer` | `5` | `False` |
| B: binary presence | `spacy_analyzer` | `2` | `True` |
| C: unigrams + bigrams | `spacy_analyzer_with_bigrams` | `2` | `False` |

The filled TODO lines are:

```python
student_X = student_vec.fit_transform(texts)
removed_terms = baseline_terms - student_terms
added_terms = student_terms - baseline_terms
student_recent = student_vec.transform(recent_texts)
student_sim = cosine_similarity(student_recent)
```

The helper below performs these same steps for each setting. The Markdown after each run gives the observed results and a worked interpretation.

## Baseline and comparison helper

The baseline is the student task’s unigram count matrix with `min_df=2`. The helper below holds speeches fixed and displays the same four comparisons as the student cell: matrix size, vocabulary changes, top terms, and cosine similarity.

In [ ]:
# Fit the shared baseline once; the only change in each run is the student vectorizer.
texts = modern['text'].astype(str)
baseline_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2)
baseline_X = baseline_vec.fit_transform(texts)


def leading_terms(vectorizer, matrix, n=15):
    """Return the n largest corpus totals with their feature names."""
    # Binary totals count speeches containing a term; count totals count occurrences.
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    return pd.Series(totals, index=vectorizer.get_feature_names_out()).sort_values(ascending=False).head(n)


def compare_option(name, student_vec):
    """Complete the scaffold's TODOs for one setting and show the comparison."""
    # STUDENT EDIT 1: learn a vocabulary and matrix for the same speeches.
    student_X = student_vec.fit_transform(texts)
    baseline_terms = set(baseline_vec.get_feature_names_out())
    student_terms = set(student_vec.get_feature_names_out())

    # STUDENT EDIT 2: set subtraction finds features unique to either version.
    removed_terms = baseline_terms - student_terms
    added_terms = student_terms - baseline_terms

    print(name.upper())
    display(pd.DataFrame({
        'version': ['baseline', name],
        'speeches': [baseline_X.shape[0], student_X.shape[0]],
        'features': [baseline_X.shape[1], student_X.shape[1]],
        'nonzero_cells': [baseline_X.nnz, student_X.nnz]
    }))
    print(f'Removed features: {len(removed_terms):,}; added features: {len(added_terms):,}')

    # These totals have a different interpretation when binary=True.
    baseline_top = leading_terms(baseline_vec, baseline_X)
    student_top = leading_terms(student_vec, student_X)
    display(pd.DataFrame({
        'baseline_term': pd.Series(baseline_top.index),
        'baseline_total': pd.Series(baseline_top.to_numpy()),
        'changed_term': pd.Series(student_top.index),
        'changed_total': pd.Series(student_top.to_numpy())
    }))

    # STUDENT EDIT 3: reuse each fitted vocabulary on the same 12 recent speeches.
    recent = modern.sort_values('year_numeric').tail(12)
    recent_texts = recent['text']
    labels = recent['doc_label'].to_numpy()
    baseline_recent = baseline_vec.transform(recent_texts)
    baseline_sim = cosine_similarity(baseline_recent)
    student_recent = student_vec.transform(recent_texts)
    student_sim = cosine_similarity(student_recent)

    # The upper triangle lists each unordered pair once and omits self-pairs.
    row, col = np.triu_indices(len(labels), k=1)
    pair_changes = pd.DataFrame({
        'speech_1': labels[row],
        'speech_2': labels[col],
        'baseline_cosine': baseline_sim[row, col],
        'changed_cosine': student_sim[row, col]
    })
    # Positive means more similar under the student's version.
    pair_changes['change'] = pair_changes['changed_cosine'] - pair_changes['baseline_cosine']
    pair_changes = pair_changes.reindex(
        pair_changes['change'].abs().sort_values(ascending=False).index
    )
    display(pair_changes.head(8).round(3))
    return {'vectorizer': student_vec, 'matrix': student_X,
            'removed': removed_terms, 'added': added_terms,
            'pair_changes': pair_changes}


In [ ]:
# Raise minimum document frequency from two speeches to five.
# A word now needs to occur in five different speeches; its within-speech count is still unchanged.
min_df_result = compare_option('min_df=5', CountVectorizer(analyzer=spacy_analyzer, min_df=5))

## min_df=5

| Measure | Baseline | min_df=5 |
| --- | --- | --- |
| Speeches | 79 | 79 |
| Features | 9124 | 5027 |
| Nonzero document-term cells | 101971 | 90947 |
| Top-10 terms shared |  | 10/10 |
| Mean absolute cosine change (66 recent pairs) |  | 0.010 |
| Mean signed cosine change |  | +0.010 |

Removed 4,097 features. Examples with relatively high baseline counts: fy, southwest, antitrust, dialogue, covenant, isis, suspension, ice, reauthorized, childcare.

Added 0 features.

Top terms in each representation:

| Rank | Baseline term | Baseline total | min_df=5 term | min_df=5 total |
| --- | --- | --- | --- | --- |
| 1 | new | 1997 | new | 1997 |
| 2 | year | 1841 | year | 1841 |
| 3 | people | 1755 | people | 1755 |
| 4 | world | 1710 | world | 1710 |
| 5 | congress | 1688 | congress | 1688 |
| 6 | america | 1597 | america | 1597 |
| 7 | years | 1405 | years | 1405 |
| 8 | government | 1335 | government | 1335 |
| 9 | american | 1299 | american | 1299 |
| 10 | federal | 1157 | federal | 1157 |

Largest cosine changes among the 12 most recent speeches:

| Speech 1 | Speech 2 | Baseline | Changed | Difference |
| --- | --- | --- | --- | --- |
| Donald Trump (2018) | Donald Trump (2020) | 0.751 | 0.766 | +0.015 |
| Donald Trump (2017) | Donald Trump (2020) | 0.745 | 0.760 | +0.015 |
| Barack Obama (2016) | Donald Trump (2018) | 0.656 | 0.671 | +0.014 |
| Barack Obama (2016) | Donald Trump (2020) | 0.605 | 0.619 | +0.014 |
| Donald Trump (2017) | Donald Trump (2018) | 0.766 | 0.780 | +0.014 |

### Worked discussion

1. **What changed?** The vocabulary lost 4,097 terms (44.9% of baseline features). The top-10 lists share 10 terms. Across the 66 recent speech pairs, mean cosine change is +0.010 (mean absolute change 0.010).
2. **Why?** A term appearing in only 2–4 speeches passes the baseline threshold but fails the new threshold. Common terms remain; less widely used terms no longer contribute to similarity.
3. **When useful?** This can reduce sparse, unstable features for a broad comparison. It may remove rare but meaningful terms, so inspect examples before treating the smaller vocabulary as better.

In [ ]:
# Count presence in each speech rather than repeated occurrences.
# Any positive within-speech count becomes 1, while the retained vocabulary stays the same.
binary_result = compare_option('binary', CountVectorizer(analyzer=spacy_analyzer, min_df=2, binary=True))

## binary

| Measure | Baseline | binary |
| --- | --- | --- |
| Speeches | 79 | 79 |
| Features | 9124 | 9124 |
| Nonzero document-term cells | 101971 | 101971 |
| Top-10 terms shared |  | 5/10 |
| Mean absolute cosine change (66 recent pairs) |  | 0.230 |
| Mean signed cosine change |  | -0.230 |

Removed 0 features.

Added 0 features.

Top terms in each representation:

| Rank | Baseline term | Baseline total | binary term | binary total |
| --- | --- | --- | --- | --- |
| 1 | new | 1997 | need | 79 |
| 2 | year | 1841 | today | 79 |
| 3 | people | 1755 | world | 79 |
| 4 | world | 1710 | nation | 79 |
| 5 | congress | 1688 | congress | 79 |
| 6 | america | 1597 | national | 79 |
| 7 | years | 1405 | great | 79 |
| 8 | government | 1335 | year | 79 |
| 9 | american | 1299 | people | 79 |
| 10 | federal | 1157 | new | 79 |

Largest cosine changes among the 12 most recent speeches:

| Speech 1 | Speech 2 | Baseline | Changed | Difference |
| --- | --- | --- | --- | --- |
| Donald Trump (2017) | Donald Trump (2018) | 0.766 | 0.449 | -0.317 |
| Barack Obama (2014) | Barack Obama (2015) | 0.810 | 0.498 | -0.312 |
| Barack Obama (2013) | Barack Obama (2014) | 0.814 | 0.503 | -0.311 |
| Donald Trump (2017) | Donald Trump (2020) | 0.745 | 0.435 | -0.310 |
| Barack Obama (2013) | Barack Obama (2015) | 0.774 | 0.467 | -0.307 |

### Worked discussion

1. **What changed?** The vocabulary is identical. For example, 'new' has 1,997 total occurrences in the baseline but appears in 79 speeches. Mean cosine change is -0.230 (mean absolute change 0.230).
2. **Why?** Every positive count becomes 1. Repeating a term many times within a speech no longer gives it extra weight; the top-term totals now measure document frequency, not token frequency.
3. **When useful?** Binary presence is apt when mentioning a concept matters more than emphasis through repetition. Repetition may be substantively important for other questions.

In [ ]:
# Keep the usual unigrams and add pairs of adjacent original tokens.
# The analyzer adds adjacent pairs before stop-word removal and retains the original unigrams.
bigram_result = compare_option('unigrams + bigrams', CountVectorizer(analyzer=spacy_analyzer_with_bigrams, min_df=2))

# Show phrase candidates separately: raw top pairs are mostly function words.
from spacy.lang.en.stop_words import STOP_WORDS
# Sum each new phrase's counts across all speeches, just as for single words.
bigram_totals = pd.Series(np.asarray(bigram_result['matrix'].sum(axis=0)).ravel(),
                          index=bigram_result['vectorizer'].get_feature_names_out())
# This is only a candidate filter; stop-word pairs can still be meaningful, especially negation.
candidate_pairs = [term for term in bigram_result['added']
                   if all(part not in STOP_WORDS for part in term.split('_', 1))]
display(bigram_totals.loc[candidate_pairs].sort_values(ascending=False).head(10).to_frame('count'))

## unigrams + bigrams

| Measure | Baseline | unigrams + bigrams |
| --- | --- | --- |
| Speeches | 79 | 79 |
| Features | 9124 | 52721 |
| Nonzero document-term cells | 101971 | 326359 |
| Top-10 terms shared |  | 6/10 |
| Mean absolute cosine change (66 recent pairs) |  | 0.058 |
| Mean signed cosine change |  | -0.054 |

Removed 0 features.

Added 43,597 features. Most frequent examples: of_the, in_the, to_the, of_our, and_the, we_have, for_the, we_must, the_world, we_will.

Frequent pair candidates without English stop words: united_states, american_people, health_care, federal_government, years_ago, social_security, soviet_union, middle_east, united_nations, economic_growth. The unfiltered top pairs above are mostly function-word sequences, so frequency alone does not establish substantive value. Negations such as 'not_good' may still matter even though they contain a stop word.

Top terms in each representation:

| Rank | Baseline term | Baseline total | unigrams + bigrams term | unigrams + bigrams total |
| --- | --- | --- | --- | --- |
| 1 | new | 1997 | of_the | 3642 |
| 2 | year | 1841 | in_the | 3020 |
| 3 | people | 1755 | new | 1997 |
| 4 | world | 1710 | year | 1841 |
| 5 | congress | 1688 | people | 1755 |
| 6 | america | 1597 | to_the | 1729 |
| 7 | years | 1405 | world | 1710 |
| 8 | government | 1335 | congress | 1688 |
| 9 | american | 1299 | of_our | 1601 |
| 10 | federal | 1157 | america | 1597 |

Largest cosine changes among the 12 most recent speeches:

| Speech 1 | Speech 2 | Baseline | Changed | Difference |
| --- | --- | --- | --- | --- |
| Barack Obama (2012) | Donald Trump (2017) | 0.678 | 0.586 | -0.091 |
| Barack Obama (2015) | Donald Trump (2017) | 0.685 | 0.593 | -0.091 |
| Barack Obama (2012) | Barack Obama (2013) | 0.792 | 0.707 | -0.084 |
| Barack Obama (2010) | Barack Obama (2012) | 0.775 | 0.692 | -0.083 |
| Barack Obama (2013) | Barack Obama (2015) | 0.774 | 0.691 | -0.083 |

### Worked discussion

1. **What changed?** The model retained the unigram features and added 43,597 adjacent-pair features. Mean cosine change is -0.054; the largest recent-pair shift is -0.091.
2. **Why?** A pair such as 'united_states' becomes a feature distinct from its two individual words. Pairs form before stop-word removal, so they reflect true adjacency. More features also change vector lengths and therefore cosine similarity.
3. **When useful?** Bigrams help when nearby word combinations matter to the research question. The most frequent pairs here are mainly function-word sequences. Inspect candidates in context and select phrases using a research question or a stated rule.

In [ ]:
# Use one shared vocabulary, then split its counts by speech period.
# Fit one vocabulary across both periods so term rates use comparable columns.
period_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2)
period_X = period_vec.fit_transform(modern['text'].astype(str))
# Assign each modern speech to 1950–79 or 1980 onward using its numeric year.
period_labels = np.where(modern['year_numeric'].to_numpy() < 1980,
                         '1950-1979', '1980 onward')

# Boolean row selection sums term occurrences separately within each period.
before_counts = np.asarray(period_X[period_labels == '1950-1979'].sum(axis=0)).ravel()
after_counts = np.asarray(period_X[period_labels == '1980 onward'].sum(axis=0)).ravel()
# Divide by all retained term occurrences in each period so unequal
# corpus sizes do not determine the comparison.
period_terms = pd.DataFrame({
    'term': period_vec.get_feature_names_out(),
    # Divide by all retained token occurrences in that period, not by speech count.
    'before_rate': before_counts / before_counts.sum(),
    'after_rate': after_counts / after_counts.sum()
})
# Positive differences mean a higher share after 1980; this is descriptive, not a causal test.
period_terms['after_minus_before'] = period_terms['after_rate'] - period_terms['before_rate']

print('More common after 1980:')
# Inspect the strongest increases; read speeches before explaining why they changed.
display(period_terms.nlargest(10, 'after_minus_before').round(5))
print('More common before 1980:')
display(period_terms.nsmallest(10, 'after_minus_before').round(5))


## Optional extension: compare periods

There are 36 speeches from 1950–79 and 43 from 1980 onward. The comparison uses the same min_df=2 unigram vocabulary in both periods. Rates are per 10,000 retained word occurrences, so the different amounts of text do not directly set the ranking.

Terms with the largest positive change (more common after 1980):

| Term | 1950–79 | 1980 onward | Difference |
| --- | --- | --- | --- |
| america | 36.63 | 85.69 | +49.06 |
| work | 26.55 | 52.35 | +25.80 |
| tonight | 9.09 | 34.78 | +25.69 |
| americans | 23.94 | 45.01 | +21.07 |
| let | 13.50 | 34.49 | +20.99 |
| children | 10.98 | 30.89 | +19.91 |
| american | 41.22 | 60.56 | +19.34 |
| jobs | 12.87 | 31.68 | +18.81 |
| know | 12.60 | 31.32 | +18.72 |
| country | 22.32 | 39.97 | +17.65 |

Terms with the largest negative change (more common in 1950–79):

| Term | 1950–79 | 1980 onward | Difference |
| --- | --- | --- | --- |
| federal | 63.00 | 32.91 | -30.09 |
| world | 81.36 | 58.04 | -23.32 |
| nations | 40.68 | 17.93 | -22.75 |
| shall | 22.77 | 0.79 | -21.98 |
| peace | 39.87 | 20.59 | -19.27 |
| program | 46.62 | 27.36 | -19.25 |
| free | 33.75 | 16.27 | -17.47 |
| programs | 39.69 | 22.75 | -16.93 |
| development | 29.07 | 14.19 | -14.88 |
| local | 22.32 | 7.78 | -14.54 |

### Worked discussion

The largest rate increases include america, work, tonight; the largest decreases include federal, world, nations. These are descriptive differences between two groups of speeches. Read speeches containing the terms before assigning a topic or historical explanation. The periods may differ in speakers, speech type, and political context as well as time.